In [2]:
import sys
import os
from pathlib import Path

# Path to your project root
project_dir = Path(r"C:\Users\dmika\DEV\Projects-local\dp100-learn")

# Change the working directory
os.chdir(project_dir)

# Add to sys.path if not already there
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

In [4]:
dataset_dir = Path(os.path.join(project_dir, "data/intel-image-classification"))
subset_dir = dataset_dir / "subset"

In [ ]:
# Download and unzip dataset from Kaggle
import kaggle
kaggle.api.authenticate()

dataset_dir = Path(os.path.join(project_dir, "data/intel-image-classification"))
dataset_dir.mkdir(parents=True, exist_ok=True)

kaggle.api.dataset_download_files('puneet6060/intel-image-classification', path=dataset_dir, unzip=True)

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification


In [15]:
# Flatten directory structure
import shutil

base = dataset_dir
for folder in ["seg_train", "seg_test", "seg_pred"]:
    inner = base / folder / folder
    if inner.exists():
        for item in inner.iterdir():
            shutil.move(str(item), str(base / folder))
        shutil.rmtree(inner)

In [ ]:
# Take a subset of the training data for quicker experiments
import random

subset_dir = dataset_dir / "subset"
subset_dir.mkdir(exist_ok=True)

for class_dir in (dataset_dir / "seg_train").iterdir():
    if class_dir.is_dir():
        files = list(class_dir.glob("*.jpg"))
        sample_files = random.sample(files, min(200, len(files)))
        target_dir = subset_dir / class_dir.name
        target_dir.mkdir(exist_ok=True)
        for f in sample_files:
            shutil.copy(f, target_dir / f.name)
print(f"Subset created at: {subset_dir}")

Subset created at: C:\Users\dmika\DEV\Projects-local\dp100-learn\data\intel-image-classification\subset


In [5]:
import json

subset_dir = dataset_dir / "subset"
jsonl_path = subset_dir / "train_annotations.jsonl"

records = []

for class_dir in subset_dir.iterdir():
    if class_dir.is_dir():
        label = class_dir.name
        for img_path in class_dir.glob("*.jpg"):
            record = {
                "image_url": str(img_path.resolve()),  # full path
                "label": label
            }
            records.append(record)

# Write to JSONL
with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print(f"✅ JSONL created at: {jsonl_path}")
print(f"Total records: {len(records)}")

✅ JSONL created at: C:\Users\dmika\DEV\Projects-local\dp100-learn\data\intel-image-classification\subset\train_annotations.jsonl
Total records: 1200
